ベース：白地図 CARTO Light No Labels

国境線：HDX OCHA 'Syrian Arab Republic - Subnational Administrative Boundaries'
https://data.humdata.org/dataset/cod-ab-syr
syr_admin0.geojson

地域線：HDX OCHA 'Syrian Arab Republic - Subnational Administrative Boundaries'
https://data.humdata.org/dataset/cod-ab-syr
syr_admin1.geojson

河川ライン：HDX Humanitarian OpenStreetMap Team 'Syria Waterways'
https://data.humdata.org/dataset/hotosm_syr_waterways
hotosm_syr_waterways_lines_geojson.zip
syr_rivers_3857.geojson

湖ポリゴン：HDX Humanitarian OpenStreetMap Team 'Syria Waterways'
https://data.humdata.org/dataset/hotosm_syr_waterways
hotosm_syr_waterways_polygons_geojson.zip
syr_lakes_4326.geojson

分析手法：GeoPandas CRS Transformation

対象地域：
・Syria (National Scale)

その他：
・河川ラインと湖ポリゴンのCRSを確認。
・河川ラインはEPSG:3857、湖ポリゴンはEPSG:4326であることを確認。
・Folium表示用に、河川ラインをEPSG:4326へ再投影。
・CRSを一致させたうえで、水系データを地図上に可視化。

In [1]:
# Core Libraries

import geopandas as gpd

import folium

/Users/marisa/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# ETL Extract
# 河川ラインデータを読み込む

rivers = gpd.read_file("syr_rivers_3857.geojson")

# Check
# 河川ラインのCRSを確認する

print(rivers.crs)

EPSG:3857


In [3]:
# ETL Extract
# 湖ポリゴンデータを読み込む

lakes = gpd.read_file("syr_lakes_4326.geojson")

# Check
# 湖ポリゴンのCRSを確認する

print(lakes.crs)

EPSG:4326


riversは　EPSG:3857、lakesは EPSG:4326、CRSが不一致

In [4]:
# ETL Transform
# 河川ラインをEPSG:4326へ再投影する

rivers_4326 = rivers.to_crs("EPSG:4326")

In [5]:
# Check
# CRSが統一されたか確認する

print(rivers_4326.crs)

print(lakes.crs)

EPSG:4326
EPSG:4326


In [6]:
# Map Setup
# 白地図ベースを作成

m = folium.Map(
    location=[34.8, 38.5], 
    zoom_start=7,
    tiles='https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}{r}.png',
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a>'
)

In [7]:
# ETL Load
# 国境線を地図へ追加
folium.GeoJson('syr_admin0.geojson', name='Country',
               style_function=lambda x: {'color': 'black', 'weight': 3, 'fillOpacity': 0}).add_to(m)

# ETL Load
# Governorate境界線を地図へ追加
folium.GeoJson('syr_admin1.geojson', name='Governorate',
               style_function=lambda x: {'color': 'gray', 'weight': 1, 'fillOpacity': 0}).add_to(m)

In [8]:
# ETL Load
# CRS変換後の河川ラインを地図へ追加

folium.GeoJson(
    rivers_4326, 
    name='Rivers',
    style_function=lambda x: {
        'color': '#5dade2', 
        'weight': 1.5, 
        'opacity': 0.7
    }
).add_to(m)

# ETL Load
# 湖ポリゴンを地図へ追加

folium.GeoJson(
    'syr_lakes_4326.geojson', 
    name='Lakes',
    style_function=lambda x: {
        'fillColor': '#5dade2', 
        'color': '#5dade2',
        'weight': 1,
        'fillOpacity': 0.5
    }
).add_to(m)

In [9]:
# ETL Load
# 隣国ラベルを地図へ追加

neighbors = {
    "TÜRKIYE": [37.5, 37.5], 
    "IRAQ": [34.5, 42.0],         
    "JORDAN": [31.9, 36.5],       
    "LEBANON": [34.2, 35.0]       
}

for name, coords in neighbors.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f'''<div style="font-size: 14pt; font-weight: bold; color: gray; white-space: nowrap; text-align: center; width: 100px; margin-left: -50px;">{name}</div>'''
        )
    ).add_to(m)

In [10]:
# ETL Load
# Governorateラベルを地図へ追加

governorates = {
    "Aleppo": [36.2, 37.5],
    "Al-Hasakeh": [36.5, 40.7],
    "Ar-Raqqa": [36.0, 39.0],
    "As-Sweida": [32.8, 36.9],
    "Daraa": [32.9, 36.2],
    "Deir-ez-Zor": [35.1, 40.5],
    "Damascus": [33.7, 36.7],
    "Hama": [35.2, 37.0],
    "Homs": [34.5, 38.3],
    "Idleb": [35.8, 36.7],
    "Lattakia": [35.6, 36.1],
    "Quneitra": [33.1, 35.9],
    "Rural Damascus": [33.5, 37.5],
    "Tartous": [34.9, 36.1]
}

for name, coords in governorates.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f'''<div style="
                font-size: 10pt; 
                color: black; 
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                width: 100px;
                margin-left: -50px;
                ">{name}</div>'''
        )
    ).add_to(m)

In [11]:
# ETL Load
# HTMLマップとして保存

m.save('17_syria_vector_crs_alighnment.html')